# Avaliação Oficial — best_placas_v2 (YOLO26s · placas BR)

Este notebook mede o desempenho **real** do modelo `best_placas_v2.pt`
no conjunto de **teste** do dataset UFPR-ALPR.

**Como usar:**
1. No painel direito do Kaggle: Settings → Accelerator → **GPU T4** (e Internet **ligada**).
2. Rode as células **na ordem**, de cima pra baixo.

Nada de tokens ou chaves neste notebook — só leitura pública do Hugging Face.


## 1) Instalar dependências

In [10]:
!pip install ultralytics huggingface_hub gdown -q

## 2) Baixar o dataset UFPR-ALPR

Baixamos direto do Google Drive (mesma fonte original usada no treino).
O `id` abaixo é o mesmo que já estava no seu notebook de fine-tuning.


In [11]:
import gdown
from pathlib import Path

CAMINHO_ZIP = "/tmp/ufpr.zip"

gdown.download(
    id="1rf4RdXoaGWCPjxfv5BGG-7mFV6otLWLO",
    output=CAMINHO_ZIP,
    quiet=False,
)
print("Download concluído.")

Downloading...
From (original): https://drive.google.com/uc?id=1rf4RdXoaGWCPjxfv5BGG-7mFV6otLWLO
From (redirected): https://drive.google.com/uc?id=1rf4RdXoaGWCPjxfv5BGG-7mFV6otLWLO&confirm=t&uuid=ca23c7a7-fcf8-433b-9492-a56412199124
To: /tmp/ufpr.zip
100%|██████████| 9.74G/9.74G [02:25<00:00, 66.9MB/s]

Download concluído.


## 3) Extrair o dataset

In [12]:
import os
import zipfile
from pathlib import Path

PASTA_UFPR = Path("/tmp/UFPR-ALPR dataset")

print("Extraindo...")
with zipfile.ZipFile(CAMINHO_ZIP, "r") as zf:
    zf.extractall(PASTA_UFPR.parent)
os.remove(CAMINHO_ZIP)
print(f"Pronto: {PASTA_UFPR} (zip apagado)")

Extraindo...
Pronto: /tmp/UFPR-ALPR dataset (zip apagado)


## 4) Montar a estrutura no formato YOLO

O dataset UFPR guarda as coordenadas da placa na linha `corners:` de cada `.txt`.
Aqui convertemos para o formato que o YOLO entende (`classe cx cy largura altura`,
com valores normalizados de 0 a 1) e organizamos em `images/` e `labels/`.

Usamos *symlink* (atalho) para as imagens em vez de copiar — é mais rápido e
economiza espaço.


In [13]:
import os
from pathlib import Path

PASTA_YOLO = Path("/tmp/dataset")

# Resolução das imagens do UFPR (fixa para todas)
LARGURA, ALTURA = 1920, 1080

# UFPR chama de training/validation/testing; o YOLO espera train/val/test
splits = {"training": "train", "validation": "val", "testing": "test"}

# Cria as pastas de destino
for split_yolo in splits.values():
    (PASTA_YOLO / "images" / split_yolo).mkdir(parents=True, exist_ok=True)
    (PASTA_YOLO / "labels" / split_yolo).mkdir(parents=True, exist_ok=True)

# Converte cada imagem + anotação
for split_ufpr, split_yolo in splits.items():
    contagem = 0
    for track in sorted((PASTA_UFPR / split_ufpr).iterdir()):
        if not track.is_dir():
            continue
        for png in sorted(track.glob("*.png")):
            txt = png.with_suffix(".txt")
            if not txt.exists():
                continue

            # cria o atalho da imagem (ignora se já existir)
            destino_img = PASTA_YOLO / "images" / split_yolo / png.name
            if not destino_img.exists():
                os.symlink(png, destino_img)

            # lê os 4 cantos da placa e converte para caixa YOLO
            label = ""
            with open(txt, encoding="utf-8") as f:
                for linha in f:
                    if linha.startswith("corners:"):
                        pares = linha.replace("corners:", "").strip().split()
                        xs = [int(p.split(",")[0]) for p in pares]
                        ys = [int(p.split(",")[1]) for p in pares]
                        cx = (min(xs) + max(xs)) / 2 / LARGURA
                        cy = (min(ys) + max(ys)) / 2 / ALTURA
                        w  = (max(xs) - min(xs)) / LARGURA
                        h  = (max(ys) - min(ys)) / ALTURA
                        label = f"0 {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}\n"
                        break

            (PASTA_YOLO / "labels" / split_yolo / (png.stem + ".txt")).write_text(label)
            contagem += 1

    print(f"  [{split_ufpr:>10}] -> {contagem} imagens processadas")

print("\nEstrutura YOLO montada com sucesso.")

  [  training] -> 1800 imagens processadas
  [validation] -> 900 imagens processadas
  [   testing] -> 1800 imagens processadas

Estrutura YOLO montada com sucesso.


## 5) Gerar o `dataset.yaml`

Arquivo de configuração que diz ao YOLO onde estão as imagens e qual é a classe.
Só temos uma classe: `license_plate` (placa).


In [14]:
CAMINHO_YAML = Path("/tmp/dataset.yaml")

CAMINHO_YAML.write_text(f"""path: {PASTA_YOLO}
train: images/train
val: images/val
test: images/test
nc: 1
names:
  0: license_plate
""")

print("dataset.yaml criado:")
print(CAMINHO_YAML.read_text())

dataset.yaml criado:
path: /tmp/dataset
train: images/train
val: images/val
test: images/test
nc: 1
names:
  0: license_plate



## 6) Conferência rápida (opcional, mas recomendado)

In [15]:
# Confere se as pastas de teste têm imagens e anotações antes de avaliar
n_imgs   = len(list((PASTA_YOLO / "images" / "test").glob("*.png")))
n_labels = len(list((PASTA_YOLO / "labels" / "test").glob("*.txt")))
print(f"Imagens de teste : {n_imgs}")
print(f"Anotações teste  : {n_labels}")
assert n_imgs > 0, "Nenhuma imagem de teste encontrada — revise as células 2 a 4."

Imagens de teste : 1800
Anotações teste  : 1800


## 7) Baixar o modelo v2 e avaliar

Baixa o `best_placas_v2.pt` publicamente do seu Hugging Face (sem token) e roda
a avaliação oficial do Ultralytics no conjunto de teste.


In [16]:
from huggingface_hub import hf_hub_download
from ultralytics import YOLO

caminho_modelo = hf_hub_download(
    repo_id="RodrigoRRC/license-plate-br-yolo26s",
    filename="best_placas_v2.pt",
)
modelo = YOLO(caminho_modelo)

metricas = modelo.val(
    data=str(CAMINHO_YAML),
    split="test",
    imgsz=1280,
)

Ultralytics 8.4.178 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLO26s summary (fused): 120 layers, 9,465,567 parameters, 0 gradients, 20.8 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 4865.4±915.3 MB/s, size: 2364.1 KB)
val: Scanning /tmp/dataset/labels/test... 1800 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1800/1800 64.4it/s 27.9s0.1ss
val: New cache created: /tmp/dataset/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 113/113 1.9it/s 59.0s0.5s
                   all       1800       1800      0.977      0.988      0.983       0.82
Speed: 2.2ms preprocess, 25.9ms inference, 0.0ms loss, 0.9ms postprocess per image
Results saved to /kaggle/working/runs/detect/val-2


## 8) Resultado final — copie e me envie

In [17]:
print("=" * 44)
print("   RESULTADO OFICIAL — best_placas_v2")
print("=" * 44)
print(f"  mAP@50      : {metricas.box.map50 * 100:.2f}%")
print(f"  mAP@50-95   : {metricas.box.map   * 100:.2f}%")
print(f"  Precisao (P): {metricas.box.mp    * 100:.2f}%")
print(f"  Recall   (R): {metricas.box.mr    * 100:.2f}%")
print("=" * 44)

   RESULTADO OFICIAL — best_placas_v2
  mAP@50      : 98.33%
  mAP@50-95   : 82.03%
  Precisao (P): 97.69%
  Recall   (R): 98.78%
